<a href="https://colab.research.google.com/github/abdelaziz2003vvb/GAN_Mnist/blob/main/First_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter

In [10]:
class Discriminator(nn.Module):
    def __init__(self, img_dim):
        super().__init__()
        self.disc = nn.Sequential(
            nn.Linear(img_dim, 128),
            nn.LeakyReLU(0.1),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.disc(x)


In [11]:
class Generator(nn.Module):
    def __init__(self, z_dim, img_dim):
        super().__init__()
        self.gen = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.1),
            nn.Linear(256, img_dim),
            nn.Tanh(),  # normalize output between -1 and 1
        )

    def forward(self, x):
        return self.gen(x)


In [12]:
# hyper parameters etc.
device = "cuda" if torch.cuda.is_available() else "cpu"
lr = 3e-4
z_dim = 64
image_dim = 28 * 28 * 1
batch_size = 32
num_epochs = 50

disc = Discriminator(image_dim).to(device)
gen = Generator(z_dim, image_dim).to(device)

fixed_noise = torch.randn((batch_size, z_dim)).to(device)

transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # IMPORTANT for Tanh generator
])

dataset = datasets.MNIST(root="dataset/", transform=transform, download=True)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

opt_disc = optim.Adam(disc.parameters(), lr=lr)
opt_gen = optim.Adam(gen.parameters(), lr=lr)

In [13]:
criterion = nn.BCELoss()
writer_fake = SummaryWriter(f"runs/GAN_MNIST/fake")
writer_real = SummaryWriter(f"runs/GAN_MNIST/real")
step = 0

for epoch in range(num_epochs):
    for batch_idx, (real, _) in enumerate(loader):
        real = real.view(-1, 784).to(device)
        batch_size = real.shape[0]

        ### -------------------------
        ### Train Discriminator
        ### -------------------------
        noise = torch.randn(batch_size, z_dim).to(device)
        fake = gen(noise)

        disc_real = disc(real).view(-1)
        lossD_real = criterion(disc_real, torch.ones_like(disc_real))

        disc_fake = disc(fake.detach()).view(-1)
        lossD_fake = criterion(disc_fake, torch.zeros_like(disc_fake))  # FIXED

        lossD = (lossD_real + lossD_fake) / 2

        disc.zero_grad()
        lossD.backward()
        opt_disc.step()

        ### -------------------------
        ### Train Generator
        ### -------------------------
        output = disc(fake).view(-1)
        lossG = criterion(output, torch.ones_like(output))  # wants D(fake)=1

        gen.zero_grad()
        lossG.backward()
        opt_gen.step()

        ### -------------------------
        ### Logging
        ### -------------------------
        if batch_idx == 0:
            print(
                f"Epoch [{epoch}/{num_epochs}] "
                f"Loss D: {lossD:.4f}, Loss G: {lossG:.4f}"
            )

            with torch.no_grad():
                fake = gen(fixed_noise).reshape(-1, 1, 28, 28)
                data = real.reshape(-1, 1, 28, 28)

                img_grid_fake = torchvision.utils.make_grid(fake, normalize=True)
                img_grid_real = torchvision.utils.make_grid(data, normalize=True)

                writer_fake.add_image("Mnist Fake Images", img_grid_fake, global_step=step)
                writer_real.add_image("Mnist Real Images", img_grid_real, global_step=step)

                step += 1

Epoch [0/50] Loss D: 0.6844, Loss G: 0.6592
Epoch [1/50] Loss D: 0.8584, Loss G: 0.6176
Epoch [2/50] Loss D: 0.3924, Loss G: 1.2245
Epoch [3/50] Loss D: 0.8082, Loss G: 0.7491
Epoch [4/50] Loss D: 0.6251, Loss G: 0.9663
Epoch [5/50] Loss D: 0.8148, Loss G: 0.7835
Epoch [6/50] Loss D: 0.6733, Loss G: 0.8512
Epoch [7/50] Loss D: 0.6020, Loss G: 1.0192
Epoch [8/50] Loss D: 0.6786, Loss G: 0.7393
Epoch [9/50] Loss D: 0.8438, Loss G: 0.9592
Epoch [10/50] Loss D: 0.6690, Loss G: 1.1784
Epoch [11/50] Loss D: 0.6929, Loss G: 0.7877
Epoch [12/50] Loss D: 0.7413, Loss G: 1.1957
Epoch [13/50] Loss D: 0.5955, Loss G: 0.8946
Epoch [14/50] Loss D: 0.6128, Loss G: 1.0985
Epoch [15/50] Loss D: 0.7278, Loss G: 0.9566
Epoch [16/50] Loss D: 0.5193, Loss G: 0.9532
Epoch [17/50] Loss D: 0.6732, Loss G: 0.9015
Epoch [18/50] Loss D: 0.7083, Loss G: 1.0316
Epoch [19/50] Loss D: 0.4779, Loss G: 1.1930
Epoch [20/50] Loss D: 0.6281, Loss G: 0.9018
Epoch [21/50] Loss D: 0.6029, Loss G: 0.8760
Epoch [22/50] Loss D